# LeetCode 1256 — Encode Number bằng Python

Link đề: https://leetcode.com/problems/encode-number/ · Độ khó: **Medium** · Chủ đề: Math, String, Bit Manipulation · 🔒 **Premium**

Skeleton LeetCode (theo lời giải Python trên doocs):

```python
class Solution:
    def encode(self, num: int) -> str:
        ...
```

> **Ghi chú Premium:** đây là bài Premium. Đề bài trong notebook được viết lại theo hiểu biết về bài (tham khảo https://leetcode.doocs.org/en/lc/1256/). Bảng mã gốc trong đề là một **hình ảnh**; bảng được tái dựng ở đây khớp với hai ví dụ chính thức (`23 → "1000"`, `107 → "101100"`). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Biết cách **suy luận quy luật** từ một bảng ví dụ (kỹ năng "đoán hàm bí mật" rất hay gặp trong phỏng vấn).
2. Hiểu thứ tự **"shortlex"** (sắp theo độ dài trước, rồi theo từ điển) của các xâu nhị phân và đếm được có bao nhiêu xâu mỗi độ dài.
3. Đi từ **liệt kê/mô phỏng** `O(num)` → **chia nhóm theo độ dài** `O(log num)` → **mẹo bit `bin(num + 1)[3:]`** `O(log num)` → **cây heap / đệ quy cha–con**.
4. Thành thạo các công cụ Python về nhị phân: `bin`, `format(x, 'b')`, `int(s, 2)`, `x.bit_length()`, toán tử dịch bit.
5. Tránh lỗi: lệch một (`num` vs `num + 1`), quên trường hợp `num = 0` (xâu rỗng), quên **đệm số 0 bên trái**.

## 2. Đề bài & ví dụ

Cho số nguyên không âm `num`, trả về **xâu mã hoá** của nó. Phép mã hoá là một "hàm bí mật" mà bạn phải tự suy ra từ bảng sau:

| `num` | mã |
|---|---|
| 0 | `""` (xâu rỗng) |
| 1 | `"0"` |
| 2 | `"1"` |
| 3 | `"00"` |
| 4 | `"01"` |
| 5 | `"10"` |
| 6 | `"11"` |
| 7 | `"000"` |

**Ví dụ 1:** `num = 23` → `"1000"`

**Ví dụ 2:** `num = 107` → `"101100"`

**Ràng buộc:** `0 <= num <= 10^9`.

Nhận xét đầu tiên: bảng đi qua **mọi xâu nhị phân**, độ dài 0, rồi độ dài 1 (`0`, `1`), rồi độ dài 2 (`00`, `01`, `10`, `11`), rồi độ dài 3…

In [1]:
import random
import time
from collections import deque
from itertools import product

TABLE = {0: "", 1: "0", 2: "1", 3: "00", 4: "01", 5: "10", 6: "11", 7: "000"}
examples = [(23, "1000"), (107, "101100")] + list(TABLE.items())

for num, want in examples[:2]:
    print(f"num = {num:4} -> {want!r}")
print("Bảng nhỏ:", TABLE)

num =   23 -> '1000'
num =  107 -> '101100'
Bảng nhỏ: {0: '', 1: '0', 2: '1', 3: '00', 4: '01', 5: '10', 6: '11', 7: '000'}


## 3. Kiến thức nền

### 3.1 Biểu diễn nhị phân: `bin`, `format`, `int(s, 2)`

- `bin(x)` trả về xâu có tiền tố `'0b'`: `bin(5) == '0b101'`.
- `format(x, 'b')` hoặc `f"{x:b}"` cho xâu **không** có tiền tố: `'101'`.
- `format(x, '0Lb')` đệm số 0 bên trái cho đủ `L` ký tự: `format(1, '03b') == '001'`.
- `int('101', 2) == 5` chuyển ngược xâu nhị phân → số.

In [2]:
x = 24
print(bin(x), format(x, 'b'), f"{x:b}", f"{x:08b}")
print("int('11000', 2) =", int('11000', 2))
print("format(1, '03b') =", format(1, '03b'))
print("Cắt chuỗi: bin(24)[2:] =", bin(24)[2:], "| bin(24)[3:] =", bin(24)[3:])

0b11000 11000 11000 00011000
int('11000', 2) = 24
format(1, '03b') = 001
Cắt chuỗi: bin(24)[2:] = 11000 | bin(24)[3:] = 1000


### 3.2 `bit_length()` và luỹ thừa của 2

`x.bit_length()` = số bit cần để viết `x` (không tính dấu, không có số 0 đầu). Với `x ≥ 1`: `2^(k-1) <= x < 2^k` ⇔ `x.bit_length() == k`.

Dịch bit: `1 << k == 2**k`, `x >> 1 == x // 2`, `x & 1 == x % 2`.

In [3]:
for x in [1, 2, 3, 4, 7, 8, 24, 108]:
    k = x.bit_length()
    print(f"x={x:4}  bin={x:b}  bit_length={k}  2^(k-1)={1 << (k-1)}  2^k={1 << k}")
print("13 >> 1 =", 13 >> 1, "| 13 & 1 =", 13 & 1)

x=   1  bin=1  bit_length=1  2^(k-1)=1  2^k=2
x=   2  bin=10  bit_length=2  2^(k-1)=2  2^k=4
x=   3  bin=11  bit_length=2  2^(k-1)=2  2^k=4
x=   4  bin=100  bit_length=3  2^(k-1)=4  2^k=8
x=   7  bin=111  bit_length=3  2^(k-1)=4  2^k=8
x=   8  bin=1000  bit_length=4  2^(k-1)=8  2^k=16
x=  24  bin=11000  bit_length=5  2^(k-1)=16  2^k=32
x= 108  bin=1101100  bit_length=7  2^(k-1)=64  2^k=128
13 >> 1 = 6 | 13 & 1 = 1


### 3.3 Liệt kê xâu nhị phân bằng `itertools.product`

`product('01', repeat=L)` sinh mọi bộ `L` ký tự từ `'01'` theo **thứ tự từ điển** — chính là thứ tự trong bảng của đề cho từng độ dài.

In [4]:
for L in range(4):
    strs = [''.join(t) for t in product('01', repeat=L)]
    print(L, len(strs), strs)

0 1 ['']
1 2 ['0', '1']
2 4 ['00', '01', '10', '11']
3 8 ['000', '001', '010', '011', '100', '101', '110', '111']


### 3.4 Đếm số xâu: tổng cấp số nhân

Có `2^L` xâu độ dài `L`. Số xâu có độ dài **nhỏ hơn** `L` là `1 + 2 + 4 + ... + 2^(L-1) = 2^L - 1`.

Vậy các xâu độ dài `L` chiếm các chỉ số `num` từ `2^L - 1` đến `2^(L+1) - 2`. Ví dụ `L = 2`: chỉ số 3..6 — khớp bảng (`3→"00"`, …, `6→"11"`).

In [5]:
for L in range(6):
    lo, hi = (1 << L) - 1, (1 << (L + 1)) - 2
    print(f"độ dài {L}: {1 << L:3} xâu, chỉ số num từ {lo:3} đến {hi:3}")

độ dài 0:   1 xâu, chỉ số num từ   0 đến   0
độ dài 1:   2 xâu, chỉ số num từ   1 đến   2
độ dài 2:   4 xâu, chỉ số num từ   3 đến   6
độ dài 3:   8 xâu, chỉ số num từ   7 đến  14
độ dài 4:  16 xâu, chỉ số num từ  15 đến  30
độ dài 5:  32 xâu, chỉ số num từ  31 đến  62


### 3.5 Hàng đợi `deque` để sinh theo chiều rộng

Sinh xâu theo thứ tự shortlex còn có thể làm bằng BFS: bắt đầu từ `""`, mỗi lần lấy một xâu `s` ra thì đẩy `s + '0'` rồi `s + '1'` vào. Thứ tự lấy ra chính là thứ tự của bảng.

In [6]:
q = deque([""])
out = []
while len(out) < 10:
    s = q.popleft()
    out.append(s)
    q.append(s + "0")
    q.append(s + "1")
print(list(enumerate(out)))

[(0, ''), (1, '0'), (2, '1'), (3, '00'), (4, '01'), (5, '10'), (6, '11'), (7, '000'), (8, '001'), (9, '010')]


## 4. Ý tưởng nền tảng / trực giác

Tính tay `num = 23`:

1. Xâu độ dài 0,1,2,3 chiếm chỉ số `0..14` (tổng `1+2+4+8 = 15` xâu).
2. Độ dài 4 chiếm chỉ số `15..30`. `23` nằm trong đoạn này ⇒ mã dài 4.
3. Vị trí trong nhóm: `23 − 15 = 8`. Viết `8` bằng 4 bit: `1000`. ⇒ mã `"1000"` ✔.

Tính tay `num = 107`: tổng xâu độ dài `≤ 5` là `63`, độ dài 6 chiếm `63..126` ⇒ offset `107 − 63 = 44 = 101100₂` (6 bit) ⇒ `"101100"` ✔.

**Một quan sát thú vị:** `23 + 1 = 24 = 11000₂` — bỏ bit `1` đầu được `1000`. `107 + 1 = 108 = 1101100₂` — bỏ bit đầu được `101100`. Trùng hợp? Không! Mục 6 sẽ giải thích.

In [7]:
for num in [23, 107, 0, 1, 2, 6, 7]:
    L = (num + 1).bit_length() - 1
    offset = num - ((1 << L) - 1)
    print(f"num={num:4}: độ dài={L}, offset={offset:3}, "
          f"num+1={num+1:b}, bỏ bit đầu -> {bin(num + 1)[3:]!r}")

num=  23: độ dài=4, offset=  8, num+1=11000, bỏ bit đầu -> '1000'
num= 107: độ dài=6, offset= 44, num+1=1101100, bỏ bit đầu -> '101100'
num=   0: độ dài=0, offset=  0, num+1=1, bỏ bit đầu -> ''
num=   1: độ dài=1, offset=  0, num+1=10, bỏ bit đầu -> '0'
num=   2: độ dài=1, offset=  1, num+1=11, bỏ bit đầu -> '1'
num=   6: độ dài=2, offset=  3, num+1=111, bỏ bit đầu -> '11'
num=   7: độ dài=3, offset=  0, num+1=1000, bỏ bit đầu -> '000'


## 5. Cách 1 — Brute Force / baseline: sinh lần lượt

**Ý tưởng:** sinh các xâu theo đúng thứ tự của bảng (BFS ở mục 3.5 hoặc `product` theo từng độ dài), dừng khi đến xâu thứ `num`.

Cách này cực kỳ trực quan và giúp **kiểm chứng** quy luật, nhưng với `num = 10^9` thì quá chậm.

In [8]:
def encode_brute(num: int) -> str:
    q = deque([""])
    i = 0
    while True:
        s = q.popleft()
        if i == num:
            return s
        i += 1
        q.append(s + "0")
        q.append(s + "1")

for num, want in examples:
    got = encode_brute(num)
    assert got == want, (num, got, want)
print("Brute force khớp mọi ví dụ:", [(n, encode_brute(n)) for n in (23, 107)])

Brute force khớp mọi ví dụ: [(23, '1000'), (107, '101100')]


Biến thể brute force không dùng hàng đợi (duyệt từng độ dài bằng `product`):

In [9]:
def encode_brute_product(num: int) -> str:
    i = 0
    L = 0
    while True:
        for t in product("01", repeat=L):
            if i == num:
                return "".join(t)
            i += 1
        L += 1

print([encode_brute_product(n) for n in range(8)])
assert all(encode_brute_product(n) == encode_brute(n) for n in range(300))

['', '0', '1', '00', '01', '10', '11', '000']


**Độ phức tạp:** sinh `num + 1` xâu, mỗi xâu dài `O(log num)` ⇒ `O(num · log num)` thời gian; hàng đợi giữ tới `O(num)` xâu ⇒ `O(num · log num)` bộ nhớ. Với `num = 10^9`: hoàn toàn không khả thi.

**Nút thắt:** ta đi qua từng xâu một, trong khi chỉ cần biết (1) mã dài bao nhiêu và (2) nó đứng thứ mấy trong nhóm — cả hai tính được bằng công thức.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chia nhóm theo độ dài.** Nhóm độ dài `L` bắt đầu tại chỉ số `2^L − 1`. Trong nhóm, các xâu xếp theo từ điển, mà xâu nhị phân **cùng độ dài** xếp theo từ điển cũng chính là xếp theo **giá trị số**. Nên xâu thứ `k` (đếm từ 0) trong nhóm = `k` viết bằng đúng `L` bit (đệm 0 bên trái).

**Insight 2 — cộng 1 là đủ.** Đặt `x = num + 1`. Nếu mã dài `L` thì `2^L − 1 <= num <= 2^(L+1) − 2` ⇔ `2^L <= x < 2^(L+1)`, tức `x` có đúng `L + 1` bit, bit đầu là `1`. Và
`x − 2^L = num + 1 − 2^L = num − (2^L − 1) = offset`.
Bỏ bit đầu của `x` chính là trừ đi `2^L`, phần còn lại là `offset` viết bằng `L` bit — **tự động có đủ số 0 đệm**!

**Quy tắc "nếu… thì…":**
- Nếu thấy bảng liệt kê "mọi xâu theo độ dài rồi từ điển" ⇒ nghĩ tới `num + 1` và bỏ bit đầu.
- Nếu `num = 0` thì `x = 1 = "1"`, bỏ bit đầu được `""` ✔.

**Insight 3 — góc nhìn cây heap.** Xếp các xâu vào cây nhị phân đầy đủ: gốc là `""` (chỉ số 0), con trái của nút `k` là `2k+1` (thêm `'0'`), con phải là `2k+2` (thêm `'1'`) — giống hệt cách đánh số mảng heap. Khi đó `encode(num) = encode((num−1)//2) + str((num−1) % 2)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chia nhóm theo độ dài + đệm 0

Từng bước:
1. Tìm `L` nhỏ nhất sao cho `num < 2^(L+1) − 1` (tăng dần `L`, hoặc dùng `bit_length`).
2. `offset = num − (2^L − 1)`.
3. Trả về `offset` viết bằng `L` bit: `format(offset, f'0{L}b')`. Cẩn thận `L = 0`: `format(0, '00b')` cho `'0'` chứ không phải `''` ⇒ xử lý riêng.

In [10]:
def encode_group(num: int) -> str:
    L = 0
    start = 0                       # chỉ số đầu tiên của nhóm độ dài L = 2^L - 1
    while num >= start + (1 << L):  # nhóm L có 2^L phần tử
        start += 1 << L
        L += 1
    if L == 0:
        return ""
    offset = num - start
    return format(offset, f"0{L}b")

print("format(0, '00b') =", repr(format(0, "00b")), "<- vì sao phải xử lý L = 0 riêng")
for num, want in examples:
    assert encode_group(num) == want, (num, encode_group(num), want)
print([encode_group(n) for n in range(10)])

format(0, '00b') = '0' <- vì sao phải xử lý L = 0 riêng
['', '0', '1', '00', '01', '10', '11', '000', '001', '010']


Bản nộp LeetCode của Cách 2:

In [11]:
class SolutionGroup:
    def encode(self, num: int) -> str:
        L = (num + 1).bit_length() - 1          # độ dài mã
        if L == 0:
            return ""
        offset = num - ((1 << L) - 1)
        return format(offset, "0%db" % L)

print([SolutionGroup().encode(n) for n, _ in examples])

['1000', '101100', '', '0', '1', '00', '01', '10', '11', '000']


**Độ phức tạp Cách 2:** vòng `while` chạy `O(log num)` lần (hoặc `O(1)` với `bit_length`), dựng xâu `O(log num)` ⇒ **`O(log num)` thời gian, `O(log num)` bộ nhớ** cho xâu kết quả.

### 7.2 Cách 3 — Mẹo bit: `bin(num + 1)[3:]` (lời giải doocs)

Theo Insight 2: `bin(num + 1)` có dạng `'0b1...'`. Cắt bỏ 3 ký tự đầu (`'0'`, `'b'`, và bit `'1'` cao nhất) là xong.

In [12]:
def encode_bin(num: int) -> str:
    return bin(num + 1)[3:]

for num in [0, 1, 2, 6, 7, 23, 107]:
    print(f"num={num:4}  bin(num+1)={bin(num + 1):>10}  ->  {encode_bin(num)!r}")

num=   0  bin(num+1)=       0b1  ->  ''
num=   1  bin(num+1)=      0b10  ->  '0'
num=   2  bin(num+1)=      0b11  ->  '1'
num=   6  bin(num+1)=     0b111  ->  '11'
num=   7  bin(num+1)=    0b1000  ->  '000'
num=  23  bin(num+1)=   0b11000  ->  '1000'
num= 107  bin(num+1)= 0b1101100  ->  '101100'


Bản nộp LeetCode (đúng như lời giải Python trên doocs):

In [13]:
class SolutionBin:
    def encode(self, num: int) -> str:
        return bin(num + 1)[3:]

assert all(SolutionBin().encode(n) == w for n, w in examples)
print("SolutionBin OK")

SolutionBin OK


**Độ phức tạp Cách 3:** `O(log num)` thời gian và bộ nhớ (độ dài xâu nhị phân ≈ 30 bit với `num ≤ 10^9`).

### 7.3 Cách 4 — Tự trích bit (không dùng `bin`)

Trong phỏng vấn đôi khi bị yêu cầu "không dùng hàm có sẵn". Ta lấy từng bit của `x = num + 1` từ thấp lên cao bằng `x & 1` và `x >>= 1`, **dừng trước** bit cao nhất (khi `x == 1`), rồi đảo ngược.

In [14]:
def encode_bits(num: int) -> str:
    x = num + 1
    bits = []
    while x > 1:               # dừng khi chỉ còn bit 1 cao nhất
        bits.append("1" if x & 1 else "0")
        x >>= 1
    return "".join(reversed(bits))

for num, want in examples:
    assert encode_bits(num) == want
print([encode_bits(n) for n in range(10)])

['', '0', '1', '00', '01', '10', '11', '000', '001', '010']


### 7.4 Cách 5 — Cây heap / đệ quy cha–con

Theo Insight 3: nút `num ≥ 1` có cha là `(num − 1) // 2`, và nó là con **trái** (`'0'`) nếu `num` lẻ, con **phải** (`'1'`) nếu `num` chẵn — tức chữ số thêm vào là `(num − 1) % 2`.

```
                 0:""
          /                \
      1:"0"                2:"1"
     /     \              /     \
 3:"00"   4:"01"      5:"10"   6:"11"
```

In [15]:
def encode_heap(num: int) -> str:
    if num == 0:
        return ""
    parent = (num - 1) // 2
    return encode_heap(parent) + str((num - 1) % 2)

def encode_heap_iter(num: int) -> str:
    out = []
    while num > 0:
        out.append(str((num - 1) % 2))
        num = (num - 1) // 2
    return "".join(reversed(out))

for num, want in examples:
    assert encode_heap(num) == want and encode_heap_iter(num) == want
print([encode_heap(n) for n in range(10)])

['', '0', '1', '00', '01', '10', '11', '000', '001', '010']


Bản nộp LeetCode của Cách 5 (vòng lặp, độ sâu chỉ ~30 nên đệ quy cũng an toàn):

In [16]:
class SolutionHeap:
    def encode(self, num: int) -> str:
        out = []
        while num:
            num -= 1
            out.append("01"[num & 1])
            num >>= 1
        return "".join(reversed(out))

assert all(SolutionHeap().encode(n) == w for n, w in examples)
print("SolutionHeap OK:", SolutionHeap().encode(10**9))

SolutionHeap OK: 11011100110101100101000000001


**Độ phức tạp Cách 4 & 5:** mỗi vòng giảm `num` khoảng một nửa ⇒ `O(log num)` vòng ⇒ **`O(log num)` thời gian và bộ nhớ**.

Thú vị: Cách 5 chính là "**hệ cơ số 2 song ánh**" (bijective numeration) dùng chữ số `{1, 2}` rồi đổi `1→'0'`, `2→'1'` — giống cách đánh tên cột Excel (`A, B, …, Z, AA, AB…`) với cơ số 26.

## 8. TRACE từng bước

**Trace Cách 3 với `num = 107`:**

| Bước | Giá trị |
|---|---|
| `x = num + 1` | `108` |
| `bin(x)` | `'0b1101100'` |
| bỏ `'0b'` | `'1101100'` (7 bit ⇒ mã dài 6) |
| bỏ bit `1` đầu | `'101100'` |

**Trace Cách 5 (heap) với `num = 23`:**

| `num` | `num − 1` | chữ số `(num−1) % 2` | cha `(num−1)//2` | xâu (đảo ngược) |
|---|---|---|---|---|
| 23 | 22 | 0 | 11 | `0` |
| 11 | 10 | 0 | 5 | `00` |
| 5 | 4 | 0 | 2 | `000` |
| 2 | 1 | 1 | 0 | `1000` |
| 0 | — | dừng | — | kết quả `"1000"` |

In [17]:
def trace_encode_heap(num: int) -> str:
    out = []
    step = 0
    print(f"{'bước':>4} | {'num':>5} | {'num-1':>5} | chữ số | {'cha':>5} | xâu hiện tại (đảo ngược)")
    while num > 0:
        step += 1
        d = (num - 1) % 2
        parent = (num - 1) // 2
        out.append(str(d))
        print(f"{step:>4} | {num:>5} | {num-1:>5} | {d:^6} | {parent:>5} | {''.join(reversed(out))}")
        num = parent
    res = "".join(reversed(out))
    print("Kết quả:", repr(res))
    return res

trace_encode_heap(23)
print()
trace_encode_heap(107)

bước |   num | num-1 | chữ số |   cha | xâu hiện tại (đảo ngược)
   1 |    23 |    22 |   0    |    11 | 0
   2 |    11 |    10 |   0    |     5 | 00
   3 |     5 |     4 |   0    |     2 | 000
   4 |     2 |     1 |   1    |     0 | 1000
Kết quả: '1000'

bước |   num | num-1 | chữ số |   cha | xâu hiện tại (đảo ngược)
   1 |   107 |   106 |   0    |    53 | 0
   2 |    53 |    52 |   0    |    26 | 00
   3 |    26 |    25 |   1    |    12 | 100
   4 |    12 |    11 |   1    |     5 | 1100
   5 |     5 |     4 |   0    |     2 | 01100
   6 |     2 |     1 |   1    |     0 | 101100
Kết quả: '101100'


'101100'

In [18]:
def trace_encode_group(num: int) -> str:
    L, start = 0, 0
    while num >= start + (1 << L):
        print(f"  độ dài {L}: chỉ số {start}..{start + (1 << L) - 1} — chưa chứa {num}, bỏ qua")
        start += 1 << L
        L += 1
    print(f"  độ dài {L}: chỉ số {start}..{start + (1 << L) - 1} — CHỨA {num}")
    offset = num - start
    res = format(offset, f"0{L}b") if L else ""
    print(f"  offset = {num} - {start} = {offset} -> viết {L} bit: {res!r}")
    return res

trace_encode_group(23)

  độ dài 0: chỉ số 0..0 — chưa chứa 23, bỏ qua
  độ dài 1: chỉ số 1..2 — chưa chứa 23, bỏ qua
  độ dài 2: chỉ số 3..6 — chưa chứa 23, bỏ qua
  độ dài 3: chỉ số 7..14 — chưa chứa 23, bỏ qua
  độ dài 4: chỉ số 15..30 — CHỨA 23
  offset = 23 - 15 = 8 -> viết 4 bit: '1000'


'1000'

## 9. Code nộp LeetCode

Đúng tên class/method/tham số như lời giải Python trên doocs (`encode(self, num: int) -> str`):

In [19]:
class Solution:
    def encode(self, num: int) -> str:
        # num + 1 có L+1 bit (bit đầu là 1); L bit còn lại chính là mã cần tìm
        return bin(num + 1)[3:]

for num, want in [(23, "1000"), (107, "101100"), (0, "")]:
    got = Solution().encode(num)
    print(num, repr(got), "OK" if got == want else "SAI")

23 '1000' OK
107 '101100' OK
0 '' OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (vị trí nhóm).** Số xâu nhị phân có độ dài `< L` là `2^0 + … + 2^(L−1) = 2^L − 1`. Vì bảng liệt kê theo độ dài tăng dần, các xâu độ dài `L` chiếm chính xác các chỉ số `2^L − 1, …, 2^(L+1) − 2`.

**Bổ đề 2 (thứ tự trong nhóm).** Với hai xâu nhị phân **cùng độ dài**, so sánh từ điển trùng với so sánh giá trị số (ký tự khác nhau đầu tiên quyết định cả hai). Nên xâu đứng thứ `k` (từ 0) trong nhóm `L` là biểu diễn `L` bit của `k`.

**Định lý.** `encode(num)` = `bin(num+1)` bỏ `'0b'` và bit đầu.
Đặt `x = num + 1`, `L = x.bit_length() − 1`. Khi đó `2^L <= x < 2^(L+1)` ⇔ `2^L − 1 <= num <= 2^(L+1) − 2` ⇒ (Bổ đề 1) mã dài `L`, offset `k = num − (2^L − 1) = x − 2^L`. Viết `x` thành `L+1` bit: bit đầu là `1` (giá trị `2^L`), `L` bit còn lại biểu diễn đúng `x − 2^L = k` **kể cả các số 0 đầu**. Theo Bổ đề 2 đó là mã cần tìm. ∎

**Cách 5 (heap).** Trong BFS sinh xâu (mục 3.5), xâu ở chỉ số `p` sinh ra hai con ở chỉ số `2p+1` (`+'0'`) và `2p+2` (`+'1'`) — chứng minh bằng quy nạp: mỗi lần lấy ra một xâu ta đẩy đúng 2 xâu, nên xâu thứ `p` đẩy các con vào vị trí `2p+1, 2p+2`. Đảo lại: cha của `num` là `(num−1)//2`, chữ số cuối là `(num−1) % 2`.

## 11. Phân tích Time / Space

Gọi `n = num`, `B = log₂(n + 1)` ≈ số bit (≤ 30 với `n ≤ 10^9`).

| Cách | Time | Space |
|---|---|---|
| 1. Sinh lần lượt (BFS) | `O(n · B)` | `O(n · B)` |
| 2. Chia nhóm + đệm 0 | `O(B)` | `O(B)` |
| 3. `bin(num+1)[3:]` | `O(B)` | `O(B)` |
| 4. Tự trích bit | `O(B)` | `O(B)` |
| 5. Heap (cha–con) | `O(B)` | `O(B)` (đệ quy: thêm `O(B)` stack) |

`O(B)` bộ nhớ là không tránh khỏi vì kết quả là một xâu dài `B` ký tự. Với ràng buộc của đề, mọi cách 2–5 coi như `O(1)` (≤ 30 thao tác).

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [20]:
ALL = {
    "group": encode_group,
    "group_submit": lambda n: SolutionGroup().encode(n),
    "bin": encode_bin,
    "bits": encode_bits,
    "heap": encode_heap,
    "heap_iter": encode_heap_iter,
    "heap_submit": lambda n: SolutionHeap().encode(n),
    "submit": lambda n: Solution().encode(n),
}
edge = [
    (0, ""),            # nhỏ nhất: xâu rỗng
    (1, "0"), (2, "1"),
    (3, "00"), (6, "11"),          # đầu & cuối nhóm độ dài 2
    (7, "000"), (14, "111"),       # đầu & cuối nhóm độ dài 3
    (15, "0000"),
    (2**30 - 2, "1" * 29),         # cuối nhóm độ dài 29
    (2**30 - 1, "0" * 30),         # đầu nhóm độ dài 30 (> 10^9 một chút, vẫn đúng)
]
for name, f in ALL.items():
    for num, want in examples + edge:
        got = f(num)
        assert got == want, (name, num, got, want)
print("Mọi cách đúng với", len(examples + edge), "test")

Mọi cách đúng với 20 test


### 12.2 Đối chiếu với brute force và tính chất

- Với `num` từ 0 đến 3000: mọi cách khớp brute force.
- Tính **song ánh**: 3000 mã đầu tiên đôi một khác nhau, và độ dài không giảm.
- Random `num` lớn (seed cố định): các cách nhanh khớp nhau, và `decode(encode(num)) == num`.

In [21]:
brute_list = [encode_brute(n) for n in range(3001)]
for name, f in ALL.items():
    assert [f(n) for n in range(3001)] == brute_list, name
assert len(set(brute_list)) == len(brute_list)
assert all(len(brute_list[i]) <= len(brute_list[i + 1]) for i in range(3000))

def decode(s: str) -> int:
    return int("1" + s, 2) - 1

rng = random.Random(1256)
for _ in range(20000):
    num = rng.randint(0, 10**9)
    vals = {f(num) for f in ALL.values()}
    assert len(vals) == 1, (num, vals)
    assert decode(vals.pop()) == num
print("0..3000 khớp brute force; 20000 số ngẫu nhiên: các cách khớp nhau & decode đúng")

0..3000 khớp brute force; 20000 số ngẫu nhiên: các cách khớp nhau & decode đúng


### 12.3 Thời gian chạy ở giới hạn

In [22]:
nums = [rng.randint(0, 10**9) for _ in range(100000)]
for name in ["bin", "group_submit", "heap_submit", "bits"]:
    f = ALL[name]
    t0 = time.perf_counter()
    for v in nums:
        f(v)
    print(f"{name:13}: 100000 lần gọi trong {time.perf_counter() - t0:.3f}s")

bin          : 100000 lần gọi trong 0.017s
group_submit : 100000 lần gọi trong 0.058s


heap_submit  : 100000 lần gọi trong 0.242s


bits         : 100000 lần gọi trong 0.242s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `bin(num)` thay vì `bin(num + 1)`

Nhìn ví dụ "nhị phân" là vội viết `bin(num)[2:]` hay `bin(num)[3:]`. Bảng bắt đầu từ xâu **rỗng** ở chỉ số 0, nên lệch một.

In [23]:
def encode_wrong1(num):
    return bin(num)[3:]          # SAI: thiếu + 1

for num, want in [(23, "1000"), (107, "101100"), (1, "0"), (0, "")]:
    got = encode_wrong1(num)
    print(f"num={num:4}: sai={got!r:10} đúng={want!r:10} {'SAI' if got != want else 'trùng hợp'}")
print("Đúng:", [Solution().encode(n) for n in (23, 107, 1, 0)])

num=  23: sai='0111'     đúng='1000'     SAI
num= 107: sai='101011'   đúng='101100'   SAI
num=   1: sai=''         đúng='0'        SAI
num=   0: sai=''         đúng=''         trùng hợp
Đúng: ['1000', '101100', '0', '']


### Lỗi 2 — Cắt `[2:]` thay vì `[3:]` (quên bỏ bit đầu)

`bin(num + 1)[2:]` vẫn còn bit `1` cao nhất ⇒ thừa một ký tự `'1'` ở đầu.

In [24]:
def encode_wrong2(num):
    return bin(num + 1)[2:]      # SAI: còn bit 1 đầu

for num, want in [(23, "1000"), (0, ""), (5, "10")]:
    print(f"num={num}: sai={encode_wrong2(num)!r}, đúng={encode_bin(num)!r}")

num=23: sai='11000', đúng='1000'
num=0: sai='1', đúng=''
num=5: sai='110', đúng='10'


### Lỗi 3 — Cách chia nhóm quên đệm số 0 bên trái

Offset `1` trong nhóm độ dài 3 phải là `"001"`, không phải `"1"`. Viết `bin(offset)[2:]` sẽ mất các số 0 đầu.

In [25]:
def encode_wrong3(num):
    L = (num + 1).bit_length() - 1
    if L == 0:
        return ""
    offset = num - ((1 << L) - 1)
    return bin(offset)[2:]       # SAI: không đệm đủ L bit

for num in [3, 7, 8, 23]:
    print(f"num={num:3}: sai={encode_wrong3(num)!r:8} đúng={encode_group(num)!r}")

num=  3: sai='0'      đúng='00'
num=  7: sai='0'      đúng='000'
num=  8: sai='1'      đúng='001'
num= 23: sai='1000'   đúng='1000'


### Lỗi 4 — `format(0, '0Lb')` với `L = 0` trả `'0'`, không phải `''`

Ngay cả khi đã đệm đúng, trường hợp `num = 0` (mã rỗng) vẫn bị sai nếu không xử lý riêng.

In [26]:
def encode_wrong4(num):
    L = (num + 1).bit_length() - 1
    offset = num - ((1 << L) - 1)
    return format(offset, f"0{L}b")   # SAI khi L = 0

print("num=0: sai =", repr(encode_wrong4(0)), "| đúng =", repr(Solution().encode(0)))
print("num=5: sai =", repr(encode_wrong4(5)), "| đúng =", repr(Solution().encode(5)), "(các trường hợp khác vẫn đúng)")

num=0: sai = '0' | đúng = ''
num=5: sai = '10' | đúng = '10' (các trường hợp khác vẫn đúng)


### Lỗi 5 — Sai ranh giới nhóm (`2^L` thay vì `2^L − 1`)

Nhầm chỉ số bắt đầu nhóm độ dài `L` là `2^L` thì toàn bộ offset lệch một.

In [27]:
def encode_wrong5(num):
    L = 0
    while num >= (1 << (L + 1)):     # SAI: ranh giới phải là 2^(L+1) - 1
        L += 1
    if L == 0:
        return "" if num == 0 else str(num - 1)
    return format(num - (1 << L), f"0{L}b")

bad = [n for n in range(40) if encode_wrong5(n) != Solution().encode(n)]
print("Số num sai trong 0..39:", len(bad), "ví dụ:", [(n, encode_wrong5(n), Solution().encode(n)) for n in bad[:4]])

Số num sai trong 0..39: 38 ví dụ: [(2, '0', '1'), (3, '1', '00'), (4, '00', '01'), (5, '01', '10')]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Bài "đoán hàm bí mật":** viết bảng ra, nhóm theo độ dài, đếm mỗi nhóm có bao nhiêu phần tử ⇒ thấy ngay `1, 2, 4, 8…`.
- **Shortlex trên bảng chữ cái `k` ký tự ⇔ hệ cơ số `k` song ánh** (bijective base-k). Với `k = 2`: `bin(num + 1)[3:]`. Với `k = 26`: tên cột Excel (bài 168).
- **Heap indexing:** cha `(i−1)//2`, con `2i+1`, `2i+2`. Bất cứ khi nào thấy "cây nhị phân đầy đủ đánh số theo tầng", nghĩ đến công thức này (bài 1261 cũng dùng!).
- Kiểm tra nhanh bằng 3 điểm: `num = 0` (rỗng), `num = 2^L − 1` (toàn `0`), `num = 2^(L+1) − 2` (toàn `1`).
- Trong Python, `bin`, `format`, `int(s, 2)`, `bit_length` xử lý số lớn tuỳ ý — không lo tràn số.

## 15. Biến thể của bài

1. **Giải mã (decode):** cho xâu `s`, tìm `num` ⇒ `int('1' + s, 2) − 1`.
2. **LeetCode 168 — Excel Sheet Column Title** và **171 — Excel Sheet Column Number:** cùng ý tưởng với cơ số 26 song ánh.
3. **Bảng chữ cái `k` ký tự tổng quát:** mã của `num` khi liệt kê mọi xâu trên `{0, …, k−1}` theo shortlex.
4. **LeetCode 1611 / Gray code (89):** một kiểu liệt kê xâu nhị phân khác (hai xâu liền nhau khác 1 bit).
5. **Tìm xâu thứ `num` có đúng `L` ký tự** (không xét các độ dài khác) ⇒ chỉ là `format(num, f'0{L}b')`.

Code minh hoạ biến thể 1 & 3:

In [28]:
def decode_num(s: str) -> int:
    return int("1" + s, 2) - 1

def encode_base_k(num: int, k: int) -> str:
    """Xâu thứ num (từ 0) trong thứ tự shortlex trên bảng chữ cái '0'..str(k-1)."""
    digits = []
    while num > 0:
        num -= 1
        digits.append(str(num % k))
        num //= k
    return "".join(reversed(digits))

print([decode_num(s) for s in ["", "0", "1", "1000", "101100"]])
print("k=3:", [encode_base_k(n, 3) for n in range(14)])
assert all(encode_base_k(n, 2) == Solution().encode(n) for n in range(2000))

[0, 1, 2, 23, 107]
k=3: ['', '0', '1', '2', '00', '01', '02', '10', '11', '12', '20', '21', '22', '000']


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sinh lần lượt | BFS/`product` theo đúng thứ tự bảng | `O(n log n)` | `O(n log n)` | Kiểm chứng quy luật, `num` nhỏ |
| 2. Chia nhóm | Tìm độ dài `L`, offset, đệm 0 | `O(log n)` | `O(log n)` | Khi muốn lập luận rõ ràng từng bước |
| 3. `bin(num+1)[3:]` | Bỏ bit đầu của `num+1` | `O(log n)` | `O(log n)` | Code nộp ngắn nhất |
| 4. Tự trích bit | `x & 1`, `x >>= 1`, dừng ở bit đầu | `O(log n)` | `O(log n)` | Khi bị cấm dùng `bin` |
| 5. Heap cha–con | `encode(n) = encode((n-1)//2) + (n-1)%2` | `O(log n)` | `O(log n)` | Tổng quát hoá sang cơ số `k` |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Không chạy code: `Solution().encode(30)` và `Solution().encode(31)` trả về gì?

### Bài 2 (tính tay)
Tính tay `encode(1000)` bằng cách chia nhóm (ghi rõ `L`, chỉ số đầu nhóm, offset).

### Bài 3 (điền TODO)
Hoàn thiện `decode_todo(s)`: trả về `num` sao cho `encode(num) == s`, **không** dùng `int(..., 2)`.

### Bài 4 (tự viết lại)
Viết lại `encode` bằng **đệ quy heap** nhưng dùng toán tử bit (`>>`, `&`) thay cho `//`, `%`: `encode_rec_bits(num)`.

### Bài 5 (mini challenge)
Viết `kth_with_prefix(num, p)`: trong dãy mã `encode(0), encode(1), …`, tìm **chỉ số nhỏ nhất** `i ≥ num` sao cho `encode(i)` bắt đầu bằng xâu `p`. (Gợi ý: nghĩ theo cây heap.)

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def decode_todo(s: str) -> int:
    x = 1                       # bit 1 "ẩn" ở đầu
    for ch in s:
        # TODO: dịch x sang trái 1 bit và cộng bit ch
        pass
    # TODO: trả về x - 1
    return None

# Bài 4 — TODO
def encode_rec_bits(num: int) -> str:
    # TODO: trường hợp cơ sở num == 0
    # TODO: cha = (num - 1) >> 1, chữ số = (num - 1) & 1
    pass

# Bài 5 — TODO
def kth_with_prefix(num: int, p: str) -> int:
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tính `num + 1`. (2) `31 = 11111₂`, `32 = 100000₂`. (3) Bỏ bit đầu: `"1111"` và `"00000"`.

**Bài 2.** (1) `1001` có bao nhiêu bit? (`2^9 = 512 <= 1001 < 1024` ⇒ 10 bit ⇒ `L = 9`). (2) Nhóm `L = 9` bắt đầu ở `2^9 − 1 = 511`. (3) Offset `1000 − 511 = 489 = 111101001₂` (9 bit).

**Bài 3.** (1) Mã là `num + 1` bỏ bit đầu ⇒ thêm lại bit `1` đầu. (2) Đọc từ trái sang: `x = x * 2 + bit`. (3) Cuối cùng trừ 1.

**Bài 4.** (1) `num == 0` ⇒ `""`. (2) `(num − 1) >> 1` là cha. (3) Ghép `encode_rec_bits(cha) + str((num − 1) & 1)`.

**Bài 5.** (1) `p` ứng với nút `q = decode(p)` trên cây heap. (2) Các nút có tiền tố `p` ở độ sâu thêm `d` tầng tạo thành một **đoạn liên tiếp** `[ (q+1)·2^d − 1, (q+1)·2^d − 1 + 2^d − 1 ]`. (3) Duyệt `d = 0, 1, 2…` và lấy giao với `[num, ∞)` — đoạn đầu tiên có phần tử `≥ num` cho đáp án.

## 19. Đáp án tham khảo

In [30]:
# Bài 1
assert Solution().encode(30) == "1111" and Solution().encode(31) == "00000"
# Bài 2
assert Solution().encode(1000) == format(1000 - 511, "09b") == "111101001"

# Bài 3
def decode_sol(s: str) -> int:
    x = 1
    for ch in s:
        x = (x << 1) | (ch == "1")
    return x - 1

# Bài 4
def encode_rec_bits_sol(num: int) -> str:
    if num == 0:
        return ""
    return encode_rec_bits_sol((num - 1) >> 1) + str((num - 1) & 1)

# Bài 5
def kth_with_prefix_sol(num: int, p: str) -> int:
    q = decode_sol(p)
    d = 0
    while True:
        lo = (q + 1) * (1 << d) - 1          # nút trái nhất có tiền tố p ở độ sâu thêm d
        hi = lo + (1 << d) - 1
        if hi >= num:
            return max(lo, num)
        d += 1

def kth_with_prefix_brute(num, p):
    i = num
    while not encode_brute(i).startswith(p):
        i += 1
    return i

rng = random.Random(99)
for _ in range(2000):
    v = rng.randint(0, 10**9)
    assert decode_sol(Solution().encode(v)) == v
    assert encode_rec_bits_sol(v) == Solution().encode(v)
for num in range(0, 300, 7):
    for p in ["", "0", "1", "01", "110", "0000"]:
        assert kth_with_prefix_sol(num, p) == kth_with_prefix_brute(num, p), (num, p)
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
# Xâu thứ num (từ 0) trong mọi xâu nhị phân theo (độ dài, từ điển)
def encode(num):  return bin(num + 1)[3:]
def decode(s):    return int('1' + s, 2) - 1

# Tổng quát: bảng chữ cái k ký tự (bijective base-k)
def encode_k(num, k):
    out = []
    while num:
        num -= 1
        out.append(num % k)
        num //= k
    return out[::-1]

# Heap indexing
parent = (i - 1) // 2;  left = 2*i + 1;  right = 2*i + 2
```

**Checklist trước khi submit:**
- [ ] `num = 0` trả về **xâu rỗng** `""`.
- [ ] Dùng `num + 1`, không phải `num`.
- [ ] Cắt `[3:]` (bỏ `'0b'` **và** bit đầu), không phải `[2:]`.
- [ ] Nếu tự đệm: đủ `L` bit, xử lý riêng `L = 0`.
- [ ] Thử nhanh 3 điểm: `6 → "11"`, `7 → "000"`, `23 → "1000"`.
- [ ] Đúng chữ ký: `class Solution: def encode(self, num: int) -> str`.